# Phase 5 — Qualitative Error Analysis

Goal: pull the sentences that are hardest for the models (high code-mixing,
wrong across most/all models), sample ~30-40 of them, and build a review table
you fill in by hand with an error `category` (sarcasm, negation scope, mixed
idiom, ambiguous-neutral, entity-heavy, transliteration noise, etc).



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/codemix_project'
DATA_DIR = f'{PROJECT_ROOT}/data'
PREDS_DIR = f'{PROJECT_ROOT}/preds'
FEATURES_DIR = f'{PROJECT_ROOT}/features'
RESULTS_DIR = f'{PROJECT_ROOT}/results'

RNG_SEED = 42  # fixed seed for the 30-40 sample, per spec
pd.set_option('display.max_colwidth', 120)


## Step 0 — Load data (mind the `sentence_id` dtype gotcha)

In [ ]:
master = pd.read_parquet(f'{DATA_DIR}/sa_master.parquet')
preds_long = pd.read_csv(f'{PREDS_DIR}/all_predictions_long.csv')

# Known gotcha: master's sentence_id is object/string, preds/features round-trip
# it as int64. Cast both to str explicitly before any join/isin.
master['sentence_id'] = master['sentence_id'].astype(str)
preds_long['sentence_id'] = preds_long['sentence_id'].astype(str)

print('master:', master.shape, master['sentence_id'].dtype)
print('preds_long:', preds_long.shape, preds_long['sentence_id'].dtype)

# Don't assume predictions cover the full corpus -- check the row count against
# what's actually in the data (models x seeds x unique sentences seen).
n_pred_sentences = preds_long['sentence_id'].nunique()
n_models_found = preds_long['model'].nunique()
n_seeds_found = preds_long['seed'].nunique()
expected_rows = n_pred_sentences * n_models_found * n_seeds_found
print(f"unique sentence_ids in preds_long: {n_pred_sentences}, models: {n_models_found}, seeds: {n_seeds_found}")
assert preds_long.shape[0] == expected_rows, (
    f"expected {n_models_found} models x {n_seeds_found} seeds x {n_pred_sentences} "
    f"sentences = {expected_rows} rows, got {preds_long.shape[0]}"
)

# preds_long is much smaller than the 15,131-row master table -- almost certainly
# because predictions only exist for the held-out test split. Confirm that here
# rather than assume it, since it changes what "the corpus" means for Step 5.
if 'split' in master.columns:
    test_ids = set(master.loc[master['split'] == 'test', 'sentence_id'])
    pred_ids = set(preds_long['sentence_id'])
    overlap = len(pred_ids & test_ids)
    print(f"\ntest-split sentence_ids in master: {len(test_ids)}")
    print(f"prediction sentence_ids also in test split: {overlap} / {len(pred_ids)}")
    if overlap == len(pred_ids):
        print("-> predictions exactly match the test split, as expected.")
    else:
        print("-> MISMATCH: predictions are not a clean subset of the test split -- investigate before proceeding.")

# Defensive: true_label/pred_label might be stored as label_id (0/1/2) rather
# than the string label. Normalize to strings using the pinned mapping if needed.
LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
if preds_long['true_label'].dtype != object:
    preds_long['true_label'] = preds_long['true_label'].map(LABEL_MAP)
if preds_long['pred_label'].dtype != object:
    preds_long['pred_label'] = preds_long['pred_label'].map(LABEL_MAP)

# Loaded once here, reused below for the modal-accuracy sanity check and the
# CMI-bucket comparison.
phase4 = pd.read_csv(f'{RESULTS_DIR}/phase4_analysis.csv')

preds_long.head()


## Step 1a — Modal prediction per model (majority vote over 3 seeds)

Uses the exact tie-break rule from Phase 4 McNemar cell —
`sub.groupby('sentence_id')['pred_label'].agg(lambda x: x.value_counts().idxmax())`
— rather than `Series.mode().iloc[0]`. On a 3-way tie (all 3 seeds disagree)
these two can pick different winners, so matching the expression exactly keeps
`n_models_wrong` here consistent with the `modal_matrix` `mcnemar_pairs.csv`
was already built from. `MODEL_NAMES` is read off the data rather than
hardcoded — sanity-checked against Phase 4's numbers in the next cell.


In [ ]:
MODEL_NAMES = sorted(preds_long['model'].unique().tolist())
print(f"Models found in preds_long: {MODEL_NAMES}")

def modal_label(sub):
    return sub.groupby('sentence_id')['pred_label'].agg(lambda x: x.value_counts().idxmax())

modal_by_model = {}
for model in MODEL_NAMES:
    modal_by_model[model] = modal_label(preds_long[preds_long['model'] == model])

modal_wide = pd.DataFrame(modal_by_model)
modal_wide.index.name = 'sentence_id'
modal_wide = modal_wide.reset_index()
modal_wide.columns = ['sentence_id'] + [f'{m}_pred' for m in MODEL_NAMES]

model_pred_cols = [f'{m}_pred' for m in MODEL_NAMES]
print(f"{len(modal_wide)} sentences x {len(model_pred_cols)} models")
modal_wide.head()


## Step 1b — Attach true label, per-model correctness, `n_models_wrong`

In [ ]:
true_labels = preds_long[['sentence_id', 'true_label']].drop_duplicates()
dupe_check = true_labels['sentence_id'].duplicated().sum()
assert dupe_check == 0, f"{dupe_check} sentence_ids have inconsistent true_label across rows"

review = modal_wide.merge(true_labels, on='sentence_id', how='left')

correct_cols = []
for pred_col in model_pred_cols:
    model_name = pred_col[:-len('_pred')]
    correct_col = f'{model_name}_correct'
    review[correct_col] = review[pred_col] == review['true_label']
    correct_cols.append(correct_col)

review['n_models_wrong'] = (~review[correct_cols]).sum(axis=1)
review['n_models_wrong'].value_counts().sort_index()


## Sanity check — modal accuracy vs. Phase 4

Since this reuses Phase 4's exact tie-break rule, per-model accuracy computed
here from `review` should match the per-model accuracy in
`results/phase4_analysis.csv` line for line. Worth eyeballing before trusting
`n_models_wrong` downstream — if these don't match, something about
`MODEL_NAMES`, the seed count, or a filter differs from Phase 4 and is worth
tracking down before sampling.


In [ ]:
print(phase4.head(10))

print("\nPer-model modal accuracy computed in this notebook:")
for model in MODEL_NAMES:
    acc = review[f'{model}_correct'].mean()
    print(f"  {model}: {acc:.4f}")

print("\nCompare the numbers above to the per-model accuracy column in phase4_analysis.csv.")


## Step 1c — Attach `cmi` and `text` from master

In [ ]:
review = review.merge(master[['sentence_id', 'cmi', 'text']], on='sentence_id', how='left')
missing_cmi = review['cmi'].isna().sum()
assert missing_cmi == 0, f"{missing_cmi} sentence_ids failed to join to master — check dtype cast"
review.head()


## Resolve CMI-bucket definition (open item from the handoff)

The Phase 4 *spec* pinned a median split (`low: cmi==0`, `mid: 0<cmi<=median`,
`high: cmi>median`), but the handoff flags that the notebook actually run may
have used tertiles (33rd/67th percentile) instead. Both are computed below —
compare the per-bucket sentence counts / accuracy against what's already in
`results/phase4_analysis.csv` and set `CMI_BUCKET_METHOD` in the next cell to
whichever one matches. 



In [ ]:
def median_split_bucket(cmi, median_val):
    if cmi == 0:
        return 'low'
    elif cmi <= median_val:
        return 'mid'
    else:
        return 'high'

def tertile_bucket(cmi, q33, q67):
    if cmi <= q33:
        return 'low'
    elif cmi <= q67:
        return 'mid'
    else:
        return 'high'

median_val = master['cmi'].median()
q33, q67 = master['cmi'].quantile([0.33, 0.67])

review['cmi_bucket_median'] = review['cmi'].apply(lambda x: median_split_bucket(x, median_val))
review['cmi_bucket_tertile'] = review['cmi'].apply(lambda x: tertile_bucket(x, q33, q67))

print('--- bucket sizes, median-split ---')
print(review['cmi_bucket_median'].value_counts())
print('\n--- bucket sizes, tertile ---')
print(review['cmi_bucket_tertile'].value_counts())

print('\nphase4_analysis.csv, for comparison (loaded back in Step 0):')
phase4.head(10)


In [ ]:
# Set this after comparing the cell above to phase4_analysis.csv:
CMI_BUCKET_METHOD = 'tertile'  # 'tertile' or 'median' -- CHANGE IF NEEDED

review['cmi_bucket'] = review[f'cmi_bucket_{CMI_BUCKET_METHOD}']
print(f"Using '{CMI_BUCKET_METHOD}' bucketing.")
review['cmi_bucket'].value_counts()


## Step 1d — Filter to high-CMI + mostly-wrong

In [ ]:
THRESHOLD = 4
candidates = review[(review['cmi_bucket'] == 'high') & (review['n_models_wrong'] >= THRESHOLD)]
print(f"n_models_wrong >= {THRESHOLD}, high CMI: {len(candidates)} sentences")

if len(candidates) < 30:
    THRESHOLD = 3
    candidates = review[(review['cmi_bucket'] == 'high') & (review['n_models_wrong'] >= THRESHOLD)]
    print(f"Relaxed to n_models_wrong >= {THRESHOLD} (still within high-CMI bucket): {len(candidates)} sentences")

print(f"Final threshold used: n_models_wrong >= {THRESHOLD}  <- note this in the write-up")


## Step 2 — Sample 30-40 (fixed seed)

In [ ]:
SAMPLE_SIZE = 40

if len(candidates) <= 40:
    sample = candidates.copy()
    print(f"Taking all {len(sample)} qualifying sentences (<=40, no sampling needed)")
else:
    sample = candidates.sample(n=SAMPLE_SIZE, random_state=RNG_SEED)
    print(f"Sampled {SAMPLE_SIZE} of {len(candidates)} qualifying sentences (seed={RNG_SEED})")

sample = sample.sort_values('n_models_wrong', ascending=False).reset_index(drop=True)
sample.head()


## Step 3 — Build and save the review table

In [ ]:
final_cols = ['sentence_id', 'text', 'cmi', 'true_label', 'n_models_wrong'] + model_pred_cols + correct_cols
review_table = sample[final_cols].copy()
review_table['category'] = ''  # fill in by hand during Step 4

out_path = f'{RESULTS_DIR}/qualitative_review_sample.csv'
review_table.to_csv(out_path, index=False)
print(f"Saved {len(review_table)} rows -> {out_path}")
review_table.head()


## Step 4 — Categorize 

read through `text` for each sampled sentence_id
and sorted them into 4-6 informal categories. Starting candidates to test the
sample against:

- **Sarcasm / irony** — sentiment flips on tone, not lexical content
- **Negation scope** — negation crosses a language switch or a long clause
- **Mixed-language idiom** — a Hindi or Hinglish idiom doesn't decompose to
  its literal token-level sentiment
- **Ambiguous / weak-signal neutral** — genuinely borderline, low-intensity
  sentiment (ties to the Phase 2.4 hint below)
- **Entity-heavy** — sentiment carried by a named entity/event the model has
  no signal for
- **Transliteration noise** — non-standard Romanized Hindi spelling breaks
  tokenization



In [ ]:
# --- Load the hand-categorized review table back from Drive ---

CATEGORIZED_PATH = f'{RESULTS_DIR}/qualitative_review_sample_categorized.csv'

review_table = pd.read_csv(CATEGORIZED_PATH)
review_table['sentence_id'] = review_table['sentence_id'].astype(str)

# Every row must carry a non-empty category.
review_table['category'] = review_table['category'].astype(str).str.strip()
blank = review_table['category'].isin(['', 'nan'])
assert blank.sum() == 0, f"{blank.sum()} rows have no category -- fill them in before continuing"

# If the sample from Step 2 is still in memory, confirm the categorized file
# matches it exactly -- guards against editing/re-saving the wrong rows.
if 'sample' in globals():
    expected_ids = set(sample['sentence_id'].astype(str))
    loaded_ids = set(review_table['sentence_id'])
    assert loaded_ids == expected_ids, (
        f"sentence_id mismatch vs the Step 2 sample: "
        f"{len(expected_ids - loaded_ids)} missing, {len(loaded_ids - expected_ids)} unexpected"
    )

print(f"Loaded {len(review_table)} categorized rows from {CATEGORIZED_PATH}")
print(f"\n{review_table['category'].nunique()} categories:")
print(review_table['category'].value_counts())
print("\ncategory x true_label:")
print(pd.crosstab(review_table['category'], review_table['true_label']))


## Step 5 — Cross-check against the Phase 2.4 'neutral is weak' hint

In [ ]:
neutral_share_sample = (review_table['true_label'] == 'neutral').mean()
neutral_share_corpus = (master['label'] == 'neutral').mean()

# Predictions (and therefore the sample) only exist for the test split, so also
# check against the test split's neutral share specifically -- the stratified
# split should keep this close to the full-corpus number, but worth confirming
# rather than assuming, especially at ~1,500 test rows.
if 'split' in master.columns:
    neutral_share_test = (master.loc[master['split'] == 'test', 'label'] == 'neutral').mean()
else:
    neutral_share_test = None

print(f"Neutral share in qualitative sample: {neutral_share_sample:.1%}")
print(f"Neutral share in full corpus:        {neutral_share_corpus:.1%}")
if neutral_share_test is not None:
    print(f"Neutral share in test split only:    {neutral_share_test:.1%}")

reference_share = neutral_share_test if neutral_share_test is not None else neutral_share_corpus
if neutral_share_sample > reference_share:
    print("\nNeutral is over-represented among hard errors relative to its share of the")
    print("population these errors were drawn from -- corroborates the Phase 2.4 per-class")
    print("F1 hint quantitatively.")
else:
    print("\nNeutral is NOT over-represented in this sample -- the per-class F1 hint may")
    print("not hold up qualitatively, or the sample is too small/skewed to tell.")

# Which failure categories the neutral errors concentrate in -- useful for the
# write-up: names the mechanism behind the over-representation, not just the count.
print("\nNeutral-true errors by category:")
print(review_table.loc[review_table['true_label'] == 'neutral', 'category'].value_counts())
